# Week 19: Time Series, State Space Models & Mamba
## Notebook 08: Crossformer: Cross-Time & Cross-Dimension Two-Stage Attention

### 1. Conceptual Intuition & Theoretical Background
While Channel Independence works well when variables are uncorrelated, complex real-world systems (e.g. stock market portfolios, multi-sensor jet turbines) have strong **cross-variable dependencies**.

Flattening time and dimensions into a single attention matrix yields:
$$O((L \times M)^2)$$
Which is computationally intractable for long sequences $L$ and many variables $M$.

**Crossformer** decouples spatio-temporal dependencies using **Two-Stage Attention (TSA)**:
1. **Cross-Time Stage (Time Attention):**
Computes self-attention across the temporal sequence dimension for each channel independently:
$$Z_{\text{time}}^{(m)} = \text{Attention}(Q_{\text{time}}^{(m)}, K_{\text{time}}^{(m)}, V_{\text{time}}^{(m)}), \quad O(M \cdot L^2)$$

2. **Cross-Dimension Stage (Router Dimension Attention):**
Computes attention across variables at each time step using learnable router tokens:
$$Z_{\text{dim}}^{(t)} = \text{Attention}(Q_{\text{dim}}^{(t)}, K_{\text{dim}}^{(t)}, V_{\text{dim}}^{(t)}), \quad O(L \cdot M^2)$$

Total complexity drops from $O(L^2 M^2)$ to $O(M L^2 + L M^2)$.

---

### 2. Scratch Python Implementation: Two-Stage Attention Block

In [ ]:
import numpy as np

class TwoStageAttentionLayerScratch:
    def __init__(self, num_vars=4, num_time_steps=8, d_model=16):
        self.num_vars = num_vars
        self.num_time_steps = num_time_steps
        self.d_model = d_model
        
    def _scaled_dot_product(self, Q, K, V):
        d_k = Q.shape[-1]
        scores = np.matmul(Q, K.swapaxes(-2, -1)) / np.sqrt(d_k)
        weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
        weights /= np.sum(weights, axis=-1, keepdims=True)
        return np.matmul(weights, V)
        
    def forward(self, X):
        """
        X: Tensor of shape (num_vars, num_time_steps, d_model)
        """
        M, L, D = X.shape
        
        # --- Stage 1: Cross-Time Attention ---
        # For each variable m in M, attend across all L time steps
        # X: (M, L, D) -> Attention on axis 1
        time_out = np.zeros_like(X)
        for m in range(M):
            # Q, K, V are linear projections of X[m]
            time_out[m] = self._scaled_dot_product(X[m], X[m], X[m])
            
        # Residual connection
        X_time = X + time_out
        
        # --- Stage 2: Cross-Dimension Attention ---
        # For each time step t in L, attend across all M variables
        # Swap axes to (L, M, D)
        X_swapped = X_time.swapaxes(0, 1)
        dim_out = np.zeros_like(X_swapped)
        for t in range(L):
            dim_out[t] = self._scaled_dot_product(X_swapped[t], X_swapped[t], X_swapped[t])
            
        # Swap back to (M, L, D) and add residual
        out = X_time + dim_out.swapaxes(0, 1)
        return out

# Test with 4 Sensors over 8 Time Segments
M_sensors = 4
L_steps = 8
D_dim = 16

mock_tsa_input = np.random.randn(M_sensors, L_steps, D_dim)

tsa = TwoStageAttentionLayerScratch(num_vars=M_sensors, num_time_steps=L_steps, d_model=D_dim)
output_repr = tsa.forward(mock_tsa_input)

print("--- Crossformer Two-Stage Attention (TSA) ---")
print(f"Input Grid:  {M_sensors} Sensors x {L_steps} Time Segments x {D_dim} Features")
print(f"Output Grid: {output_repr.shape}")
print("Explicit cross-temporal and cross-sensor dependencies captured without O((L*M)^2) explosion!")

### 3. Step-by-Step Code Walkthrough
1. **Stage 1 (Cross-Time)**: Models temporal dynamics within each sensor time series independently.
2. **Stage 2 (Cross-Dimension)**: Interchanges axes to evaluate cross-sensor correlations at each timestamp.
3. **Decoupled Factorization**: Avoids constructing the full $(M \times L)^2$ joint attention matrix.

---

### 4. Industry Applications & Trade-offs
- **Aerospace & Industrial IoT:** Detecting sensor anomalies where failures manifest as deviations across correlated telemetry channels.
- **Trade-off:** High memory access overhead when transposing multi-dimensional tensors.

---

### 5. Top Interview Questions & Answers
**Q1: Why is joint 2D attention over both time and dimensions computationally infeasible?**
> **Answer:** If there are $M$ variables and $L$ time steps, flattening them into tokens results in sequence length $M \times L$. Full self-attention takes $O((M \cdot L)^2) = O(M^2 L^2)$, which quickly causes out-of-memory errors.

**Q2: How does Crossformer's Two-Stage Attention resolve this quadratic bottleneck?**
> **Answer:** It decomposes attention into two sequential 1D passes: first across time ($O(M L^2)$), then across dimensions ($O(L M^2)$), achieving linear scaling with respect to the other dimension.